# AI Conduit - Wan2.2 動画生成

## 使い方
1. ColabのSecretsに`GITHUB_TOKEN`を設定
2. 上から順に実行
3. 自動でGitHubにアップロードされます

In [ ]:
# GPU確認
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3}GB')


In [ ]:
!pip install diffusers transformers accelerate -q
print('インストール完了')


In [ ]:
import torch, os
from diffusers import WanPipeline
from diffusers.utils import export_to_video

pipe = WanPipeline.from_pretrained('Wan-AI/Wan2.1-T2V-1.3B-Diffusers', torch_dtype=torch.float16)
pipe = pipe.to('cuda')

scenes = [
    ('hook', 'frustrated developer staring at dark terminal screen cinematic 4K'),
    ('why', 'developer typing frantically at computer with error messages on screen'),
    ('solution', 'developer smiling at computer screen showing success green terminal'),
    ('step1', 'close up hands typing commands dark mechanical keyboard terminal screen'),
    ('step2', 'code editor screen YAML configuration file dark theme syntax highlighting'),
    ('result', 'developer celebrating at desk successful code deployment multiple monitors'),
    ('cta', 'smartphone screen showing download notification modern app close up'),
]

os.makedirs('/content/cogvideo', exist_ok=True)
for scene_name, prompt in scenes:
    print(f'生成中: {scene_name}...')
    output = pipe(prompt=prompt, num_frames=49, num_inference_steps=20, guidance_scale=5.0)
    export_to_video(output.frames[0], f'/content/cogvideo/{scene_name}.mp4', fps=8)
    size = os.path.getsize(f'/content/cogvideo/{scene_name}.mp4') // 1024
    print(f'  OK {scene_name}.mp4 ({size}KB)')

print('全シーン生成完了！')


In [ ]:
# GitHubにアップロード
# ColabのSecretsからGITHUB_TOKENを取得
from google.colab import userdata
import requests, base64, os

GH_TOKEN = userdata.get('GITHUB_TOKEN')
h = {'Authorization': f'token {GH_TOKEN}'}

for f in sorted(os.listdir('/content/cogvideo')):
    path = f'/content/cogvideo/{f}'
    with open(path, 'rb') as fp:
        content = base64.b64encode(fp.read()).decode()
    r0 = requests.get(f'https://api.github.com/repos/aiconduit/ai-conduit-pipeline/contents/assets/cogvideo/{f}', headers=h)
    sha = r0.json().get('sha', '') if r0.status_code == 200 else ''
    data = {'message': f'Auto: Wan2.2 {f}', 'content': content}
    if sha: data['sha'] = sha
    r = requests.put(f'https://api.github.com/repos/aiconduit/ai-conduit-pipeline/contents/assets/cogvideo/{f}', headers=h, json=data)
    print(f'{f}: {r.status_code}')

print('完了！')
